# Hybrid Retrieval + Reranking on the OrchestrAI Synthetic Enterprise Corpus

This notebook extends the vector-database pipeline with **two retrieval improvements** while keeping the rest of the logic aligned with the earlier notebooks:

1. **Hybrid retrieval** in Milvus, which combines:
   - dense semantic embeddings from Ollama
   - sparse BM25-style keyword matching generated inside Milvus

2. **Reranking** with a cross-encoder, which reorders the hybrid-retrieved candidate chunks before they are sent to the LLM.

The purpose of this notebook is to create a fourth comparison point that isolates the effect of **hybrid recall + reranked precision** on the same synthetic enterprise corpus.

## What changes compared with the other notebooks?

### `rag_basic`
- in-memory document store
- simpler retrieval pipeline
- no vector database
- no reranking

### `rag_vectordb`
- Milvus-backed dense retrieval
- same overall prompting logic as the basic notebook
- better persistence and retrieval scalability
- still no reranking

### `rag_reranker`
- dense Milvus retrieval
- reranking added after retrieval
- improves the ordering of dense-retrieved candidates

### This notebook: `rag_hybrid_rerank`
- Milvus-backed **hybrid retrieval** (dense + BM25)
- reranking added after hybrid retrieval
- same answer-generation constraints as the other notebooks
- useful when exact terminology matters **and** semantic meaning matters

## Environment requirements

This notebook assumes:

- Python **3.12** in the project virtual environment
- Ollama is running locally at `http://localhost:11434`
- the Ollama embedding model `mxbai-embed-large` is available
- the Ollama chat model `llama3.1` is available
- Milvus Standalone is running locally at `http://localhost:19530`
- the `dummy_data/` folder is present in the project root

This version uses Milvus built-in BM25 support, so it is intended for a real Milvus server setup rather than a lightweight local-only storage mode.

## Imports and runtime checks

In [1]:
import sys
import warnings
import time
from pathlib import Path
import requests

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

warnings.filterwarnings("ignore")

print("Python:", sys.executable)
print("Ollama tags status:", requests.get("http://localhost:11434/api/tags").status_code)

import importlib
import results_logger
importlib.reload(results_logger)
from results_logger import save_result_row, clear_results_for_implementation
clear_results_for_implementation("hybrid_rerank")

from haystack import Pipeline
from haystack.dataclasses import ChatMessage
from haystack.components.builders import ChatPromptBuilder
from haystack.components.writers import DocumentWriter
from haystack.components.preprocessors import DocumentSplitter
from haystack.components.rankers import SentenceTransformersSimilarityRanker

from milvus_haystack import MilvusDocumentStore, MilvusHybridRetriever
from milvus_haystack.function import BM25BuiltInFunction

from haystack_integrations.components.embedders.ollama import (
    OllamaDocumentEmbedder,
    OllamaTextEmbedder,
)
from haystack_integrations.components.generators.ollama import OllamaChatGenerator

from docling_haystack.converter import DoclingConverter
from docling.chunking import HybridChunker

print("Imports work")

Python: /Users/nikos/Thesis/.venv/bin/python
Ollama tags status: 200
Cleared existing rows for implementation='hybrid_rerank'
Imports work


## Connect to Milvus

This notebook uses a dedicated Milvus collection for the **hybrid + reranking** variant.

The collection schema differs from dense-only retrieval because it stores:

- a dense vector field for semantic search
- a sparse vector field generated through BM25
- the source text field used by the built-in sparse function

A new collection name is used to avoid mixing schemas between experiments.

In [2]:
COLLECTION_NAME = "rag_hybrid_rerank_docs"

document_store = MilvusDocumentStore(
    connection_args={"uri": "http://localhost:19530"},
    collection_name=COLLECTION_NAME,
    vector_field="vector",
    sparse_vector_field="sparse_vector",
    text_field="text",
    builtin_function=[
        BM25BuiltInFunction(
            input_field_names="text",
            output_field_names="sparse_vector",
        )
    ],
    drop_old=True,
)

print("Milvus hybrid + rerank collection ready:", COLLECTION_NAME)

Milvus hybrid + rerank collection ready: rag_hybrid_rerank_docs


## Load the OrchestrAI dataset

The corpus is organized by department under `./dummy_data`, and all `.docx` files are loaded recursively.

This keeps the corpus exactly aligned with the earlier notebooks so the comparison remains fair.

In [ ]:
DOCUMENTS_DIR = PROJECT_ROOT / "data" / "dummy_data"
FILES = [str(f.resolve()) for f in DOCUMENTS_DIR.rglob("*.docx")]

print("Files found:", len(FILES))
for f in FILES[:10]:
    print("-", Path(f).name)

Files found: 54
- VAL-SALES-003_Campaign_Approval_Workflow_refreshed.docx
- VAL-SALES-004_CRM_Field_Governance_Note_refreshed.docx
- VAL-SALES-005_Discount_Exception_Policy_refreshed.docx
- VAL-SALES-001_Lead_Routing_Rules_refreshed.docx
- VAL-SALES-002_Webinar_Launch_Brief_refreshed.docx
- VAL-SEC-003_Patch_Compliance_Reminder.docx
- VAL-SEC-004_Data_Retention_Notice.docx
- VAL-SEC-005_Third_Party_Access_Approval_Standard.docx
- VAL-SEC-002_Privileged_Access_Review_Memo.docx
- VAL-SEC-001_Phishing_Alert_Bulletin.docx


## Indexing strategy

The indexing logic stays close to the dense Milvus notebook:

1. Convert each DOCX file with `DoclingConverter`
2. Preserve explicit metadata such as:
   - `file_path`
   - `file_name`
   - `department`
3. Split the converted content into retrieval-sized chunks
4. Generate dense embeddings with Ollama
5. Write the chunks into Milvus

The **hybrid** part is handled by the document-store schema. Milvus builds the sparse BM25 representation from the stored text automatically.

In [4]:
OLLAMA_EMBED_MODEL = "mxbai-embed-large"
HF_TOKENIZER_ID = "mixedbread-ai/mxbai-embed-large-v1"

converter = DoclingConverter(
    chunker=HybridChunker(
        tokenizer=HF_TOKENIZER_ID,
        max_tokens=300,
    )
)

doc_embedder = OllamaDocumentEmbedder(
    model=OLLAMA_EMBED_MODEL,
    url="http://localhost:11434"
)

writer = DocumentWriter(document_store=document_store)

def normalize_meta(meta):
    cleaned = {}
    for key, value in (meta or {}).items():
        if key == "_split_overlap":
            continue
        if isinstance(value, (str, int, float, bool)) or value is None:
            cleaned[key] = value
        else:
            cleaned[key] = str(value)
    return cleaned

stored_chunks = 0
indexing_start = time.perf_counter()

splitter = DocumentSplitter(
    split_by="word",
    split_length=80,
    split_overlap=20
)

for i, path in enumerate(FILES, 1):
    print(f"[{i}/{len(FILES)}] Indexing {Path(path).name}")

    converted = converter.run(paths=[path])["documents"]

    for doc in converted:
        meta = normalize_meta(doc.meta)
        meta["file_path"] = path
        meta["file_name"] = Path(path).name
        meta["department"] = Path(path).parent.name
        doc.meta = meta

    split_docs = splitter.run(documents=converted)["documents"]

    for doc in split_docs:
        meta = normalize_meta(doc.meta)
        meta["file_path"] = path
        meta["file_name"] = Path(path).name
        meta["department"] = Path(path).parent.name
        doc.meta = meta

    embedded_docs = doc_embedder.run(documents=split_docs)["documents"]
    writer.run(documents=embedded_docs)

    stored_chunks += len(embedded_docs)

indexing_time_s = round(time.perf_counter() - indexing_start, 4)

print("Indexing completed.")
print("Stored chunks in Milvus:", stored_chunks)
print("Document store count:", document_store.count_documents())
print("Indexing time (s):", indexing_time_s)

[1/54] Indexing VAL-SALES-003_Campaign_Approval_Workflow_refreshed.docx


Calculating embeddings:   0%|                                                                     | 0/1 [00:00<?, ?it/s]I0504 16:11:44.698699 1391948 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(81, generation: 1)
I0504 16:11:44.698763 1391948 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(81, generation: 1)
I0504 16:11:44.698796 1391948 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(81, generation: 1)
I0504 16:11:44.698799 1391948 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(81, generation: 1)
I0504 16:11:44.698800 1391948 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(81, generation: 1)
I0504 16:11:44.698801 1391948 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(81, generation: 1)
I0504 16:11:44.698802 1391948 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(81, generation: 1)
I0504 16:11:44.698803 1391948 ev_poll_posix.cc:593] FD from fork parent still in p

[2/54] Indexing VAL-SALES-004_CRM_Field_Governance_Note_refreshed.docx


Calculating embeddings: 100%|█████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  5.27it/s]


[3/54] Indexing VAL-SALES-005_Discount_Exception_Policy_refreshed.docx


...

[53/54] Indexing TEAMQA-LOG-001_OrchestrAI_Logistics_and_Field_Fulfillment_Handbook.docx


Calculating embeddings: 100%|█████████████████████████████████████████████████████████████| 2/2 [00:01<00:00,  1.51it/s]


[54/54] Indexing TEAMQA-PROD-001_OrchestrAI_Product_and_Program_Management_Handbook.docx


Calculating embeddings: 100%|█████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.40s/it]

Indexing completed.
Stored chunks in Milvus: 654
Document store count: 654
Indexing time (s): 34.0511


## Build the hybrid + reranking RAG pipeline

The answer-generation logic remains the same as in the other notebooks:
- use only the retrieved context
- do not invent missing information
- cite source filenames when possible

The retrieval logic now has two stages:

1. **Hybrid retrieval**
   - combines dense similarity and sparse keyword matching
   - returns a candidate pool

2. **Reranking**
   - uses a cross-encoder to rescore the candidate pool against the query
   - forwards only the top reranked chunks to the prompt builder

In [5]:
template = [
    ChatMessage.from_user(
        """
Respond to the User Query using only the provided Context.

General Guidelines:
- Use only the provided context.
- If the answer is not found in the context, say so clearly.
- Do not make up facts, document sections, pages, or citations.
- If the answer comes from multiple documents, cite all relevant documents.
- Respond in the same language as the user’s query.
- Do not use emojis.
- Be professional and concise.
- Do not write a conclusion or follow-up unless the user asks for it.
- If the context contains a direct policy rule, exception, SLA, or escalation instruction, state it directly and do not hedge.
- Prefer the most specific procedural source over broader handbook summaries when both are present.
- If the retrieved context does not explicitly contain the rule, exception, owner, or next step needed to answer the question, say that the retrieved context is insufficient instead of inferring or guessing.
- Do not mention system names, tools, workflows, or teams unless they are explicitly present in the retrieved context.

Citation Rules:
- For every important claim, cite the source document filename.
- If available in the retrieved context, also mention the document ID or department.
- Do not mention source chapter, section, or page unless they are explicitly present in the retrieved context.

Context:
{% for document in documents %}
[Source: {{ document.meta.get("file_name", document.meta.get("file_path", "unknown source")) }}]
{{ document.content }}

{% endfor %}

Question: {{question}}
Answer:
"""
    )
]

RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

text_embedder = OllamaTextEmbedder(
    model=OLLAMA_EMBED_MODEL,
    url="http://localhost:11434"
)

retriever = MilvusHybridRetriever(
    document_store=document_store,
    top_k=12
)

reranker = SentenceTransformersSimilarityRanker(
    model=RERANK_MODEL,
    top_k=3
)

chat_generator = OllamaChatGenerator(
    model="llama3.1",
    url="http://localhost:11434",
    generation_kwargs={
        "temperature": 0.0,
        "top_p": 1.0,
        "seed": 42,
    }
)
prompt_builder = ChatPromptBuilder(
    template=template,
    required_variables=["question", "documents"]
)

reranker.warm_up()

hybrid_rerank_pipeline = Pipeline()
hybrid_rerank_pipeline.add_component("text_embedder", text_embedder)
hybrid_rerank_pipeline.add_component("retriever", retriever)
hybrid_rerank_pipeline.add_component("reranker", reranker)
hybrid_rerank_pipeline.add_component("prompt_builder", prompt_builder)
hybrid_rerank_pipeline.add_component("llm", chat_generator)

hybrid_rerank_pipeline.connect("text_embedder.embedding", "retriever.query_embedding")
hybrid_rerank_pipeline.connect("retriever.documents", "reranker.documents")
hybrid_rerank_pipeline.connect("reranker.documents", "prompt_builder.documents")
hybrid_rerank_pipeline.connect("prompt_builder.prompt", "llm.messages")

The CrossEncoder `tokenizer_kwargs` argument was renamed and is now deprecated. Please use `processor_kwargs` instead.
Loading weights: 100%|██████████████████████████████████████████████████████████████| 105/105 [00:00<00:00, 6562.10it/s]


🚅 Components
  - text_embedder: OllamaTextEmbedder
  - retriever: MilvusHybridRetriever
  - reranker: SentenceTransformersSimilarityRanker
  - prompt_builder: ChatPromptBuilder
  - llm: OllamaChatGenerator
🛤️ Connections
  - text_embedder.embedding -> retriever.query_embedding (list[float])
  - retriever.documents -> reranker.documents (List[Document])
  - reranker.documents -> prompt_builder.documents (list[Document])
  - prompt_builder.prompt -> llm.messages (list[ChatMessage])

## Helper function for hybrid + reranked queries

The function below:
- embeds the question
- performs hybrid retrieval using both the dense embedding and the raw query text
- reranks the retrieved candidate chunks
- prints the reranked sources and snippets
- runs the final answer-generation pipeline
- returns both the reranked documents and the final answer

The return values make later comparisons easier across `rag_basic`, `rag_vectordb`, `rag_reranker`, and this notebook.

In [6]:
def run_hybrid_rerank_query(question, retriever_top_k=12, reranker_top_k=3):
    print("=" * 80)
    print("QUESTION:")
    print(question)
    print("=" * 80)

    retrieval_start = time.perf_counter()

    q_emb = text_embedder.run(text=question)["embedding"]

    retr_out = retriever.run(
        query_embedding=q_emb,
        query_text=question,
        top_k=retriever_top_k,
    )
    candidate_docs = retr_out["documents"]

    rerank_out = reranker.run(
        query=question,
        documents=candidate_docs,
        top_k=reranker_top_k,
    )
    top_docs = rerank_out["documents"]

    retrieval_time_s = round(time.perf_counter() - retrieval_start, 4)

    print(
        f"\nRERANKED DOCUMENTS: {len(top_docs)} "
        f"(from {len(candidate_docs)} hybrid-retrieved candidates)"
    )
    print(f"RETRIEVAL + RERANK TIME (s): {retrieval_time_s}\n")

    for i, d in enumerate(top_docs, 1):
        source = d.meta.get("file_name", d.meta.get("file_path", "unknown source"))
        department = d.meta.get("department", "unknown department")
        snippet = (d.content or "")[:400].replace("\n", " ")
        print(f"[{i}] Source: {source} | Department: {department}")
        print(f"    {snippet}...")
        print()

    generation_start = time.perf_counter()

    result = hybrid_rerank_pipeline.run({
        "text_embedder": {"text": question},
        "retriever": {"query_text": question, "top_k": retriever_top_k},
        "reranker": {"query": question, "top_k": reranker_top_k},
        "prompt_builder": {"question": question},
    })

    answer = result["llm"]["replies"][0].text
    generation_time_s = round(time.perf_counter() - generation_start, 4)

    print("=" * 80)
    print("FINAL ANSWER:")
    print("=" * 80)
    print(answer)
    print(f"\nGENERATION TIME (s): {generation_time_s}")

    return top_docs, answer, retrieval_time_s, generation_time_s

In [7]:
def run_and_save_hybrid_rerank(question, scenario_id, retriever_top_k=12, reranker_top_k=3):
    top_docs, answer, retrieval_time_s, generation_time_s = run_hybrid_rerank_query(
        question,
        retriever_top_k=retriever_top_k,
        reranker_top_k=reranker_top_k,
    )

    save_result_row(
        implementation="hybrid_rerank",
        scenario_id=scenario_id,
        question=question,
        generated_answer=answer,
        top_docs=top_docs,
        retriever_top_k=retriever_top_k,
        reranker_top_k=reranker_top_k,
        indexing_time_s=indexing_time_s,
        retrieval_time_s=retrieval_time_s,
        generation_time_s=generation_time_s,
    )

    return top_docs, answer

## Evaluation scenarios

The following five questions are kept identical to the earlier notebooks so that the systems can be compared directly.

The scenarios increase in complexity:

1. Basic fact retrieval
2. Procedural retrieval
3. Multi-team operational reasoning
4. Policy and exception handling
5. Cross-functional enterprise reasoning

### Scenario 1 — Basic fact retrieval
This is the clean baseline. It checks whether the system can retrieve one precise operational fact and cite the right supporting source.

In [8]:
top_docs_hybrid_rerank_s1, answer_hybrid_rerank_s1 = run_and_save_hybrid_rerank(
    'What is the first-response SLA for a Sev-1 support incident?',
    scenario_id="hr1",
    retriever_top_k=12,
    reranker_top_k=3,
)

QUESTION:
What is the first-response SLA for a Sev-1 support incident?

RERANKED DOCUMENTS: 3 (from 12 hybrid-retrieved candidates)
RETRIEVAL + RERANK TIME (s): 0.6168

[1] Source: TEAMQA-CS-001_OrchestrAI_Customer_Success_and_Support_Handbook.docx | Department: Q&A Hanbooks
    Customer Success and Support Handbook Specialised Q&A **Q: Who owns customer communication during a Sev-1 incident?** **A:** The assigned support lead owns the operational updates, but the Customer Success Manager owns stakeholder alignment and executive communication. If the issue touches release risk or hardware failure, Engineering or Logistics may join the bridge. **Q: What is the target first...

[2] Source: TEAMQA-CS-001_OrchestrAI_Customer_Success_and_Support_Handbook.docx | Department: Q&A Hanbooks
    Customer Success and Support Handbook Service targets and escalation guide Sev-1 first response, **Target** = 15 min. Sev-1 first response, **Reviewed by** = Support manager. Sev-1 first response, **Notes

### Scenario 2 — Procedural retrieval
This scenario tests whether the system can reconstruct a process made of several required steps rather than a single isolated fact.

In [9]:
top_docs_hybrid_rerank_s2, answer_hybrid_rerank_s2 = run_and_save_hybrid_rerank(
    'What steps must be completed before a new employee is considered ready for day one at OrchestrAI?',
    scenario_id="hr2",
    retriever_top_k=12,
    reranker_top_k=3,
)

QUESTION:
What steps must be completed before a new employee is considered ready for day one at OrchestrAI?

RERANKED DOCUMENTS: 3 (from 12 hybrid-retrieved candidates)
RETRIEVAL + RERANK TIME (s): 0.3343

[1] Source: VAL-HR-001_Day_One_Onboarding_Checklist.docx | Department: HR
    Day-One Onboarding Checklist **Document ID:** VAL-HR-001 **Owner:** HR & People Operations **Confidentiality:** Internal Use Only Purpose: This checklist standardizes the minimum tasks that must be complete before an OrchestrAI employee can be considered ready for day-one onboarding. **Department**, **Value** = HR & People Operations. **Effective date**, **Value** = 2026-02-24. **Applies to**, **V...

[2] Source: VAL-HR-001_Day_One_Onboarding_Checklist.docx | Department: HR
    Day-One Onboarding Checklist Day-one completion rules **•** The new hire must be able to sign in with baseline credentials and MFA. **•** Payroll and tax forms must be complete or formally escalated. **•** The manager must confirm fi

### Scenario 3 — Multi-team operational reasoning
This scenario requires combining information across more than one operational area to determine ownership and next actions.

In [10]:
top_docs_hybrid_rerank_s3, answer_hybrid_rerank_s3 = run_and_save_hybrid_rerank(
    'A new engineer starts on Monday, but their laptop has not arrived and their required access is still pending. Which teams are involved and what should happen next?',
    scenario_id="hr3",
    retriever_top_k=12,
    reranker_top_k=3,
)

QUESTION:
A new engineer starts on Monday, but their laptop has not arrived and their required access is still pending. Which teams are involved and what should happen next?

RERANKED DOCUMENTS: 3 (from 12 hybrid-retrieved candidates)
RETRIEVAL + RERANK TIME (s): 0.3536

[1] Source: VAL-IT-001_Laptop_Provisioning_FAQ.docx | Department: IT
    Common Questions **Q: How far in advance should a laptop be requested?** A: Standard new-hire laptop requests should be present in the hiring workflow at least seven business days before the employee start date. International shipments, executive hires, and custom software bundles may require ten to fifteen business days. **Q: Which team starts the request?** A: The request begins with the hiring ...

[2] Source: VAL-IT-001_Laptop_Provisioning_FAQ.docx | Department: IT
    1 business day. Non-standard software or admin rights request, **Primary Owner** = Endpoint Engineering. Non-standard software or admin rights request, **Escalation Trigger** = 

### Scenario 4 — Policy and exception handling
This scenario checks whether the system can distinguish a normal policy from an approved exception path under operational pressure.

In [11]:
top_docs_hybrid_rerank_s4, answer_hybrid_rerank_s4 = run_and_save_hybrid_rerank(
    'Can OrchestrAI make an urgent hardware purchase without following the normal PO workflow if a warehouse outage is at risk?',
    scenario_id="hr4",
    retriever_top_k=12,
    reranker_top_k=3,
)

QUESTION:
Can OrchestrAI make an urgent hardware purchase without following the normal PO workflow if a warehouse outage is at risk?

RERANKED DOCUMENTS: 3 (from 12 hybrid-retrieved candidates)
RETRIEVAL + RERANK TIME (s): 0.3132

[1] Source: TEAMQA-SALES-001_OrchestrAI_Sales_and_Solutions_Handbook.docx | Department: Q&A Hanbooks
    opportunity has a documented warehouse problem, a credible business sponsor, a timeline that matches operational constraints, a measurable business case, and enough process detail to determine whether OrchestrAI can support the workflow without hidden delivery risk. **Q: When must a Solutions Consultant join a deal?** **A:** A Solutions Consultant must join when the opportunity involves integratio...

[2] Source: VAL-IT-005_Password_Reset_Escalation_SOP.docx | Department: IT
    Purpose and scope This standard operating procedure defines how OrchestrAI IT handles password reset requests that cannot be completed through the normal self-service flow. It appl

### Scenario 5 — Cross-functional enterprise reasoning
This is the most demanding scenario because it requires synthesizing support, product, logistics, and customer-success information into one answer.

In [12]:
top_docs_hybrid_rerank_s5, answer_hybrid_rerank_s5 = run_and_save_hybrid_rerank(
    'A customer renewal is at risk because support cases are recurring, a requested feature is still unavailable, and replacement hardware is delayed. What should OrchestrAI do next, and which teams must be involved?',
    scenario_id="hr5",
    retriever_top_k=12,
    reranker_top_k=3,
)

QUESTION:
A customer renewal is at risk because support cases are recurring, a requested feature is still unavailable, and replacement hardware is delayed. What should OrchestrAI do next, and which teams must be involved?

RERANKED DOCUMENTS: 3 (from 12 hybrid-retrieved candidates)
RETRIEVAL + RERANK TIME (s): 0.605

[1] Source: VAL-CS-004_Renewal_Risk_Signals_Brief.docx | Department: Customer Support
    Renewal Risk Signals Brief **Document ID:** VAL-CS-004 **Department:** Customer Success & Support **Confidentiality:** Internal Use Only **Effective date**, Director, Customer Success Operations = 2026-02-10. **Effective date**, **Document type** = **Review date**. **Effective date**, Risk brief = 2026-08-10 Purpose: Defines the most important signals used by OrchestrAI Customer Success teams to i...

[2] Source: TEAMQA-CS-001_OrchestrAI_Customer_Success_and_Support_Handbook.docx | Department: Q&A Hanbooks
    and quarterly business reviews once the account is operating steadily. **Q:

### Ground Truth CSV
Create the rag_ground_truth.csv file that will contain the manual ground truth results.

In [13]:
from ground_truth_builder import build_ground_truth_sheet

build_ground_truth_sheet()

Created rag_ground_truth.csv with 25 rows.


PosixPath('rag_ground_truth.csv')